# NB39 - trimmed trailing scores on the full archive, in event time

NB38 screened trimmed trailing scores against forward Sharpe on the dense-polling regime only
(from 2026-04-01): about four months, 70 overlapping decisions. This notebook runs the same
question on the whole archive from mid-2025, which means running it on WEEKLY data for most of
its length: through 2025 the archive holds about one mark per vault per week, one every two days
in January-March 2026, and a mark on most days from April (this notebook keeps one last mark
per UTC day, cell 2).

Forward-filling weekly marks to a daily grid and then computing daily statistics is where the
gotchas live, so nothing here is computed on a daily grid. Every score uses observed marks
only: event log returns between consecutive marks inside the window, trimming by a FRACTION of
events rounded up (so a weekly vault with 13 events in 90 days loses 2 or 4 of them and a daily
vault with 90 loses 9 or 23), staleness measured in days since the vault's own last mark, and forward
outcomes that must contain marks near the end of the window. The primary horizon is 60 days
because a 30-day window holds about four weekly marks; the 30-day horizon is kept as a
secondary target. The three polling regimes are screened together and separately.

**Focus is forward Sharpe.** Verdict DIAGNOSTIC: a screen, not a result. No vault is selected,
masked or tuned by name. **Headline: on 192 decisions over a year, trailing risk-adjusted
scores are positively associated with the next 60 days' Sharpe - the 180-day Sharpe and Sortino
lead at rho 0.26, and 15 of 16 signals clear the computed simultaneous bound with
60-day blocks, 15 with 90-day blocks and 15 with 180-day blocks - and no trim improvement
is detected.** No block choice is both long enough to cover the 180-day persistence of the
trailing scores and numerous enough for reliable controlled bootstrap inference (the 192
decisions are 2.13 180-day block-equivalents), so the bounds are the computed figures under
each block choice, not a controlled family-wise result; NB38's four-month, 30-day-horizon screen could not resolve any
of this.

**Based on:** [38-research-trimmed-return-screen.ipynb](38-research-trimmed-return-screen.ipynb)
(machinery and its two reviews), [33-research-lead-comparison.ipynb](33-research-lead-comparison.ipynb)
(the archive density table that defines the regimes). Snapshot `vault-prices.parquet`
255,548,076 bytes, sha256 `11e7c5e0103e1012`, last mark 2026-09-16 (cell 2).

## Method

Marks: one per vault per UTC day (the last poll of the day). Events: consecutive marks; event
return = log price ratio; event span = days between them. A candidate at decision T needs, in
the trailing window (T-1-W, T-1] for W in 90 and 180 days, at least 8 event returns (9 marks), a
mark at or before the window start, its first in-window mark within 14 days of that start and
its last mark within 14 days of T-1 (so marks bracket the window and the observed event span is
between W - 28 and W days), and a TVL of at least 7,500 USD at the last mark. Scores per window: return score
(sum of event returns, annualised over W; raw and with the best 10% and 25% of events removed),
Sharpe score (return score over event volatility sqrt(sum r^2 / W x 365), raw and trimmed the
same way), Sortino, event volatility. Forward outcomes over (T, T + H] for H = 60 (primary) and
30: log return from the mark carried at T to the last mark in the window, event volatility,
event Sharpe, log max drawdown on the mark path; a window needs at least 6 (H = 60) or 4
(H = 30) marks and one within 14 days of its end. Panel: 29,912 candidate-dates, 397
vaults, 192 decisions 2025-07-01 to 2026-07-18, 68.5% of rows under 360 days old measured from
the vault's first mark anywhere in the archive (cell 4);
59,998 candidate-dates dropped as stale or under-marked, 25,200 for TVL, 858
for an unobserved forward window (cell 4).
Event returns per 90-day window, median: 12 in the weekly regime, 31 in
the transition, 89 in the dense regime; marks per 60-day forward window 9 /
60 / 60 (cell 4, decision-date regimes).

Inference as NB38 with one change forced by the horizon: per-date signed Spearman averaged
over dates, one two-way cluster bootstrap of TILED, non-wrapping 30-decision (60-day)
date blocks (a random offset per draw, every decision in exactly one tile, whole tiles resampled
with replacement and never truncated; measured coverage of the first and last block of dates
1.015 and 1.000 of the mean against 0.997 for the middle, cell 6) x vault clusters, 500 draws, seed 20260917, shared across every hypothesis,
studentised max-T simultaneous lower bounds over the 16-signal family on the primary target
(critical 2.28), the same with 45-decision (90-day) tiles (critical
2.39) and 90-decision (180-day) tiles (critical 2.46; the
source tiling is one or two full tiles plus remainders depending on the offset, and a draw
repeats tiles), paired trimmed-minus-raw differences on the same draws
with their own family bound, and a foresight-oracle reachability assertion (lower bound
0.996, cell 8). Regime cohorts contain only decisions whose whole 60-day horizon lies
inside the regime (crosses: 60, dense Apr 2026 on: 55, transition Jan-Mar 2026: 15, weekly 2025: 62 decisions).

## Key new insights and what did we learn from this experiment?

**1. On a year of data, trailing risk-adjusted scores are positively associated with the next
60 days' Sharpe.** With 60-day blocks, 15 of 16 signals clear the computed simultaneous lower
bound of zero on forward 60-day Sharpe; with 90-day blocks the same 15, and with 180-day
tiles - the longest trailing window, 2.13 block-equivalents in the archive - 15 (cell 6). The strongest are
the 180-day Sharpe and Sortino scores: `sharpe180_f10` 0.257 (bounds
0.136 / 0.125 / 0.123 at 60 / 90 / 180-day blocks), `sortino180`
0.255 (0.129 / 0.127 / 0.122), `sharpe180_f00`
0.251 (0.127 / 0.125 / 0.120). The 180-day Sharpe and
Sortino scores correlate with forward RETURN at 0.191 to
0.193 and with 30-day forward Sharpe at 0.212 to
0.224. NB38 saw 0.136 for the 180-day Sharpe on 70 decisions at a 30-day horizon and
could not clear a family bound (its manifest values are displayed in cell 6); this screen has three times the
decisions and a horizon that holds enough marks. Going from 60- to 90-day tiles leaves the
passing set unchanged and moves individual bounds in both directions by at most
0.016; the 180-day tiles cover the trailing scores' own persistence but the archive holds only
2.13 of them, so their bounds are descriptive, not a robustness proof.

**2. No trim improvement is detected under a family bound.** Paired comparisons are MATCHED: raw
and trimmed Spearmans are computed per date on the joint finite mask of both scores and the
target, so a difference is trimming alone (cell 6). The return-score trim at 90 days lifts the
matched correlation from 0.151 to 0.211 (difference
0.060 [-0.011, 0.136], add-one p
0.116 alone, simultaneous bound over the 8 paired comparisons
-0.025); at 180 days the difference is 0.010. As in NB38
the trimmed return score takes on a volatility loading (signed correlation with lower forward
volatility 0.126 raw, 0.504 trimmed) and lands where the raw 180-day
scores already are. The Sharpe-score trim differences are -0.012 and
0.006 at 10%, -0.036 and
-0.031 at 25%, with intervals that include zero (cell 6). The old-cohort
90-day 25% trim reads -0.092 [-0.242,
0.025] on its own interval (cell 10), one of several cohort comparisons and
exploratory. Nothing here establishes that trimming helps or harms; it establishes that no
improvement was found.

**3. Regime cohorts with the forward outcome contained in the regime.** The forward 60 days lie
inside the regime; the trailing 180-day score of an early dense-regime decision still reads
pre-April marks, so this is outcome-containment, not a wholly within-regime comparison. Weekly
2025 (62 decisions whose whole horizon is in 2025): `sharpe180_f00` 0.281 (bound
0.081), `sortino180` 0.278. The transition cohort has too few contained decisions to screen (15).
Dense April on (55 decisions): `sharpe180_f00` 0.204 (bound -0.001).
(cell 10). The weekly-regime estimate is the largest. Two readings are consistent with that and
this screen cannot separate them: quality persisted more in 2025's universe (which was
76.1% under a year old, cell 4), or coarse, irregularly observed returns - a weekly mark is a
snapshot, and the return between two of them is a week's interval return - together with
trailing scores that barely change between marks make trailing and forward scores mechanically
more alike than daily marks would.

**4. Young and old vaults: the 180-day risk-adjusted scores carry over to both; the return
scores only to the young.** Age is measured from the vault's first mark anywhere in the
archive (68.5% of rows under 360 days; 76.1% in the weekly cohort; cell 4).
Young vaults (192 decisions): `sharpe180_f00` 0.241 (bound
0.086), `ret90_f10` 0.232 (bound 0.112), 15 of 16
clear. Old vaults (192 decisions): `sharpe180_f00` 0.238 (bound
0.033), `sortino180` 0.241, and only 3 clear
(`sharpe180_f00`, `sharpe180_f10`, `sortino180`); the return scores sit at 0.120-0.122 with
bounds below zero (cell 10). The two are estimates on different samples, not a test of a
difference. The incumbent's 360-day CAGR leg cannot score a vault younger than a year; the
scores that carry over in both cohorts need 180 days.

**5. What this says about the incumbent's ranker.** Its Sortino leg looks back 45 days and its
CAGR leg 360; this screen has no 45-day window, so the leg itself is not tested, but the pattern
is that 180-day risk-adjusted scores (0.251-0.255) sit
above 90-day ones (0.201-0.212) and above raw
return scores at either length. That is a lead for a ranker test, not a result: the effect on a
six-name book is what the standing gates measure, and portfolio consequences are not claimed
here.

## Summary of results

Forward 60-day Sharpe screen, all regimes (cell 6): signed Spearman, simultaneous lower bounds
over the 16-signal family with 60-day blocks (critical 2.28) and 90-day blocks (critical
2.39), unadjusted one-sided add-one p; the forward volatility column is signed so positive =
the score's good end had LOWER forward volatility.

| signal | rho fwd60 Sharpe | bound, 60-d blocks | bound, 90-d blocks | p | rho fwd60 return | rho fwd60 vol | rho fwd30 Sharpe |
|---|---|---|---|---|---|---|---|
| ret90_f00 | 0.151 | 0.025 | 0.023 | 0.008 | 0.138 | 0.126 | 0.111 |
| ret90_f10 | 0.211 | 0.095 | 0.095 | 0.004 | 0.199 | 0.504 | 0.179 |
| ret90_f25 | 0.209 | 0.090 | 0.091 | 0.004 | 0.203 | 0.579 | 0.178 |
| sharpe90_f00 | 0.212 | 0.090 | 0.088 | 0.004 | 0.153 | 0.145 | 0.172 |
| sharpe90_f10 | 0.199 | 0.078 | 0.069 | 0.004 | 0.146 | 0.174 | 0.169 |
| sharpe90_f25 | 0.176 | 0.043 | 0.028 | 0.004 | 0.136 | 0.255 | 0.155 |
| sortino90 | 0.201 | 0.082 | 0.081 | 0.004 | 0.147 | 0.151 | 0.158 |
| vol90 | 0.160 | 0.050 | 0.053 | 0.002 | 0.167 | 0.675 | 0.141 |
| ret180_f00 | 0.203 | 0.077 | 0.070 | 0.004 | 0.181 | 0.260 | 0.163 |
| ret180_f10 | 0.213 | 0.084 | 0.078 | 0.002 | 0.201 | 0.522 | 0.179 |
| ret180_f25 | 0.196 | 0.064 | 0.061 | 0.002 | 0.189 | 0.594 | 0.163 |
| sharpe180_f00 | 0.251 | 0.127 | 0.125 | 0.002 | 0.192 | 0.257 | 0.214 |
| sharpe180_f10 | 0.257 | 0.136 | 0.125 | 0.002 | 0.193 | 0.280 | 0.224 |
| sharpe180_f25 | 0.219 | 0.098 | 0.082 | 0.002 | 0.181 | 0.349 | 0.184 |
| sortino180 | 0.255 | 0.129 | 0.127 | 0.002 | 0.191 | 0.271 | 0.212 |
| vol180 | 0.110 | -0.024 | -0.026 | 0.028 | 0.122 | 0.612 | 0.098 |

Matched paired trimmed-minus-raw on forward 60-day Sharpe (cell 6; both Spearmans on the joint
finite mask per date): per-comparison 95% intervals and add-one p; simultaneous lower bounds
over each 8-comparison family are all below zero.

| | 10% of events removed | 25% of events removed |
|---|---|---|
| ret score, 90 d | 0.060 [-0.011, 0.136] p 0.116 | 0.058 [-0.025, 0.149] p 0.168 |
| ret score, 180 d | 0.010 [-0.050, 0.077] p 0.778 | -0.007 [-0.088, 0.076] p 0.830 |
| sharpe score, 90 d | -0.012 [-0.062, 0.035] p 0.547 | -0.036 [-0.116, 0.039] p 0.327 |
| sharpe score, 180 d | 0.006 [-0.048, 0.063] p 0.834 | -0.031 [-0.127, 0.056] p 0.439 |

Per cohort, `sharpe180_f00` on forward 60-day Sharpe (cell 10): weekly 2025
0.281 (bound 0.081), dense 0.204 (-0.001),
young 0.241 (0.086), old 0.238 (0.033).

## Robustness of results

- Nothing is computed on a daily grid: scores and outcomes are event-time on observed marks
  inside their windows, eligibility needs a mark within 14 days of T-1 and 8 events in the
  window, forward windows need 6 (60 d) or 4 (30 d) marks and one within 14 days of the end
  (cell 4). A weekly vault's forward outcome is a 60-day return over about nine snapshots, and
  its "Sharpe" is a coarse quantity built from interval returns between them.
- The screen is reachable: a foresight oracle clears the 17-signal bound at
  0.996 (cell 8).
- Date blocks are tiled 30-decision (60-day) blocks with a random offset per draw, no wrapping,
  whole tiles never truncated; measured inclusion of the first / last block of dates is
  1.015 / 1.000 of the mean at 60-day tiles, 1.026 / 1.047 at 90,
  1.001 / 1.002 at 180 (each end asserted within 0.15 of 1.0), against
  0.997 / 0.968 / 0.980 for the middle (cell 6). The passing set is 15 / 15 / 15 across the three. The 180-day tiles
  match the longest trailing window but the archive holds 2.13 of them, so no block choice
  here is both long enough for the dependence and numerous enough for a well-behaved bootstrap;
  the bounds are the computed figures under each choice and are not claimed as controlled
  family-wise evidence.
  192 decisions over about a year hold roughly six non-overlapping 60-day horizons.
- One bootstrap per screen; the regime and cohort screens are separate samples with separate
  bootstraps and their intervals are not comparable across screens in a paired sense. Regime
  cohorts contain only decisions whose whole horizon lies inside the regime.
- Trimmed scores are ranking transformations, not investable returns; forward max drawdown is
  in log units.
- The panel is the archive, not the incumbent's candidate pool; no portfolio claim is made.


## Part 0. Archive, provenance, constants, regimes


In [1]:
import hashlib, json, math
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import rankdata
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 200)

ARCHIVE = Path.home() / ".cache/tradingstrategy/vaults/downloads/vault-prices.parquet"
raw_bytes = ARCHIVE.read_bytes()
PROVENANCE = {"file": str(ARCHIVE), "bytes": len(raw_bytes), "sha256": hashlib.sha256(raw_bytes).hexdigest()}
del raw_bytes
HYPERCORE_CHAIN = 9999

WINDOWS = (90, 180)
TRIM_FRACTIONS = (0.0, 0.10, 0.25)
HORIZONS = {60: 6, 30: 4}          # forward horizon days -> minimum observed marks in the window
PRIMARY_H = 60
PANEL_START = pd.Timestamp("2025-07-01")
DECISION_STEP_DAYS = 2
MIN_TVL_USD = 7_500.0
MIN_EVENTS = 8        # events strictly inside the window, i.e. at least 9 marks
STALE_DAYS = 14
MIN_CANDIDATES = 8
MIN_DATES = 40
YOUNG_DAYS = 360
DRAWS = 500
#: Date blocks must be at least as long as the forward horizon (60 days = 30 decisions) so that
#: overlapping outcomes stay together inside a block; blocks are NOT wrapped circularly, because
#: joining July 2026 to July 2025 would splice two polling regimes. 45 decisions (90 days) is an
#: intermediate sensitivity; the 180-day score persistence is covered only by DATE_BLOCK_LONG.
DATE_BLOCK = 30
DATE_BLOCK_SENSITIVITY = 45
#: 90 decisions = 180 days, the longest trailing-score window. With 192 decisions this leaves
#: about two blocks per draw, so its bounds are as much a statement about the sample size as
#: about the dependence; it is run because no shorter block covers the score persistence.
DATE_BLOCK_LONG = 90
SEED = 20260917
LEVEL = 0.95
REGIMES = [("weekly 2025", pd.Timestamp("2025-01-01"), pd.Timestamp("2026-01-01")),
           ("transition Jan-Mar 2026", pd.Timestamp("2026-01-01"), pd.Timestamp("2026-04-01")),
           ("dense Apr 2026 on", pd.Timestamp("2026-04-01"), pd.Timestamp("2027-01-01"))]


def regime_of(t):
    """Regime of the decision date."""
    for name, a, b in REGIMES:
        if a <= t < b:
            return name
    return "other"


def regime_contained(t, H):
    """Regime that contains BOTH the decision date and its whole forward horizon, else 'crosses'."""
    for name, a, b in REGIMES:
        if a <= t and t + pd.Timedelta(days=H) < b:
            return name
    return "crosses"


df = pd.read_parquet(ARCHIVE, columns=["address", "chain", "share_price", "total_assets", "name"])
df = df[df["chain"] == HYPERCORE_CHAIN].reset_index()
df["timestamp"] = pd.to_datetime(df["timestamp"])
# Vault age is measured from the vault's FIRST mark anywhere in the archive, before any date cut.
FIRST_MARK = df.groupby("address")["timestamp"].min().dt.floor("D")
df = df[df["timestamp"] >= pd.Timestamp("2025-01-01")].sort_values(["address", "timestamp"])
df["date"] = df["timestamp"].dt.floor("D")
marks = df.groupby(["address", "date"])[["share_price", "total_assets"]].last().reset_index()
marks = marks[marks["share_price"] > 0]
names = df.groupby("address")["name"].last()
LAST_MARK = df["timestamp"].max()
display(pd.Series({**PROVENANCE, "last_mark": str(LAST_MARK), "hypercore_vaults": marks["address"].nunique(),
                   "mark_days": len(marks)}, name="value").to_frame())

# Polling density by month on the mark-day grid: marks per vault per day among vaults above the TVL floor.
mm = marks[marks["total_assets"] >= MIN_TVL_USD].copy()
mm["month"] = mm["date"].dt.to_period("M")
dens = mm.groupby("month").agg(vaults=("address", "nunique"), mark_days=("date", "size"))
dens["mark_days_per_vault_per_day"] = dens["mark_days"] / dens["vaults"] / 30.0
display(dens.round(3).T)


,value
file,/Users/moo/.cache/tradingstrategy/vaults/downl...
bytes,255548076
sha256,11e7c5e0103e10125a27fcb2ed58febd394cab3e317771...
last_mark,2026-09-16 07:25:06.705000
hypercore_vaults,604
mark_days,112919


month,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09
vaults,57.000,65.000,73.000,81.000,92.000,101.000,133.000,153.000,172.000,197.000,211.000,230.000,257.000,300.000,341.000,322.00,332.000,320.00,297.000,304.000,271.000
mark_days,210.000,220.000,257.000,332.000,314.000,344.000,543.000,520.000,597.000,835.000,742.000,1023.000,2795.000,6831.000,8279.000,8307.00,8806.000,8157.00,8433.000,8091.000,4046.000
mark_days_per_vault_per_day,0.123,0.113,0.117,0.137,0.114,0.114,0.136,0.113,0.116,0.141,0.117,0.148,0.363,0.759,0.809,0.86,0.884,0.85,0.946,0.887,0.498


## Part 1. The event-time panel

One row per (decision date, vault). Nothing is forward-filled: a window's statistics come from
the marks inside it, and a vault whose last mark is older than 14 days is not a candidate.


In [2]:
def event_scores(r: np.ndarray, W: int) -> dict:
    """Raw and trimmed return and Sharpe scores from event returns `r` in a window of W days.
    Trimming removes the ceil(f x n) largest event returns; the window length stays the denominator,
    so these are ranking scores, not investable returns."""
    out = {}
    n = len(r)
    order = np.sort(r)
    for f in TRIM_FRACTIONS:
        k = int(math.ceil(f * n)) if f > 0 else 0
        kept = order[:n - k] if k else order
        rate = float(kept.sum() * 365.0 / W)
        vol = float(math.sqrt((kept ** 2).sum() * 365.0 / W)) if len(kept) else float("nan")
        tag = f"f{int(f * 100):02d}"
        out[f"ret_{tag}"] = rate
        out[f"sharpe_{tag}"] = rate / vol if vol and vol > 0 else float("nan")
    downside = math.sqrt((np.clip(r, None, 0.0) ** 2).sum() * 365.0 / W)
    out["sortino"] = out["ret_f00"] / downside if downside > 0 else float("nan")
    out["vol"] = float(math.sqrt((r ** 2).sum() * 365.0 / W))
    out["events"] = int(n)
    return out


def forward_outcome(carried: float, fwd_prices: np.ndarray, H: int) -> dict:
    """Outcome from the mark carried at T to the marks in (T, T + H]."""
    prices = np.concatenate([[carried], fwd_prices])
    r = np.diff(np.log(prices))
    total = float(r.sum())
    vol = float(math.sqrt((r ** 2).sum() * 365.0 / H))
    path = np.concatenate([[0.0], np.cumsum(r)])
    return {f"fwd{H}_return": total, f"fwd{H}_vol": vol,
            f"fwd{H}_sharpe": (total * 365.0 / H) / vol if vol > 0 else float("nan"),
            f"fwd{H}_log_max_dd": float(np.min(path - np.maximum.accumulate(path))),
            f"fwd{H}_events": int(len(fwd_prices))}


max_h = max(HORIZONS)
last_decision = LAST_MARK.floor("D") - pd.Timedelta(days=max_h)
decisions = pd.date_range(PANEL_START, last_decision, freq=f"{DECISION_STEP_DAYS}D")
rows = []
dropped = {"stale_or_too_few_marks": 0, "tvl": 0, "forward_marks": 0}
for address, g in marks.groupby("address"):
    g = g.set_index("date").sort_index()
    mdays = g.index
    prices = g["share_price"].to_numpy()
    tvls = g["total_assets"].to_numpy()
    born = FIRST_MARK[address]
    for t in decisions:
        t1 = t - pd.Timedelta(days=1)
        i_last = int(mdays.searchsorted(t1, side="right")) - 1   # last mark at or before T-1
        if i_last < 0 or (t1 - mdays[i_last]).days > STALE_DAYS:
            dropped["stale_or_too_few_marks"] += 1
            continue
        if tvls[i_last] < MIN_TVL_USD:
            dropped["tvl"] += 1
            continue
        row = {"address": address, "date": t, "age_days": int((t - born).days), "regime": regime_of(t),
               "days_since_mark": int((t1 - mdays[i_last]).days), "first_mark_before_2025": bool(born < pd.Timestamp("2025-01-01"))}
        scored_any = False
        for W in WINDOWS:
            start = t1 - pd.Timedelta(days=W)
            i_first = int(mdays.searchsorted(start, side="right"))   # first mark strictly after start
            n_marks = i_last - i_first + 1
            # The vault must have existed before the window (a mark at or before its start), so a
            # W-day score is never computed on a vault younger than W days.
            # The score must SPAN the window: a mark at or before the window start, the first
            # in-window mark within STALE_DAYS of the start, and the last within STALE_DAYS of T-1
            # (already required). Otherwise a W-day score could be eight clustered late events.
            if n_marks < MIN_EVENTS + 1 or i_first == 0 or (mdays[i_first] - start).days > STALE_DAYS:
                for f in TRIM_FRACTIONS:
                    tag = f"f{int(f * 100):02d}"
                    row[f"ret{W}_{tag}"] = np.nan; row[f"sharpe{W}_{tag}"] = np.nan
                row[f"sortino{W}"] = np.nan; row[f"vol{W}"] = np.nan; row[f"events{W}"] = int(max(n_marks, 0))
                continue
            # Event returns between consecutive marks INSIDE the window only: the first event starts
            # at the first mark in the window, so no return interval begins before the window.
            seg = prices[i_first:i_last + 1]
            r = np.diff(np.log(seg))
            s = event_scores(r, W)
            for key, value in s.items():
                base, _, tag = key.partition("_")
                row[f"{base}{W}_{tag}" if tag else f"{base}{W}"] = value
            scored_any = True
        if not scored_any:
            dropped["stale_or_too_few_marks"] += 1
            continue
        i_carry = int(mdays.searchsorted(t, side="right")) - 1
        carried = float(prices[i_carry])
        ok_any = False
        for H, min_marks in HORIZONS.items():
            t_end = t + pd.Timedelta(days=H)
            j0 = int(mdays.searchsorted(t, side="right")); j1 = int(mdays.searchsorted(t_end, side="right"))
            fwd = prices[j0:j1]
            if len(fwd) < min_marks or (t_end - mdays[j1 - 1]).days > STALE_DAYS:
                for k in ("return", "vol", "sharpe", "log_max_dd"):
                    row[f"fwd{H}_{k}"] = np.nan
                row[f"fwd{H}_events"] = int(len(fwd))
                continue
            row.update(forward_outcome(carried, fwd, H))
            ok_any = True
        if not ok_any:
            dropped["forward_marks"] += 1
            continue
        rows.append(row)
panel = pd.DataFrame(rows)
panel["young"] = panel["age_days"] < YOUNG_DAYS
panel["regime_contained"] = [regime_contained(t, PRIMARY_H) for t in panel["date"]]
print("candidate-dates dropped:", dropped)
print(f"panel: {len(panel):,} rows, {panel['address'].nunique()} vaults, {panel['date'].nunique()} decisions "
      f"{panel['date'].min().date()} to {panel['date'].max().date()}; young (< {YOUNG_DAYS} d from the vault's first mark in "
      f"the whole archive) share of rows {panel['young'].mean():.1%}; rows from vaults first marked before 2025: "
      f"{panel['first_mark_before_2025'].mean():.1%}")
by_regime = panel.groupby("regime").agg(rows=("address", "size"), vaults=("address", "nunique"), decisions=("date", "nunique"),
                                        events90_median=("events90", "median"), events180_median=("events180", "median"),
                                        fwd60_events_median=("fwd60_events", "median"), fwd30_events_median=("fwd30_events", "median"),
                                        fwd60_finite=("fwd60_sharpe", lambda s: float(np.isfinite(s).mean())),
                                        fwd30_finite=("fwd30_sharpe", lambda s: float(np.isfinite(s).mean())),
                                        young_share=("young", "mean"))
display(by_regime.round(3))

SIGNALS = []
for W in WINDOWS:
    for f in TRIM_FRACTIONS:
        tag = f"f{int(f * 100):02d}"
        SIGNALS.append({"name": f"ret{W}_{tag}", "direction": "high", "window": W, "trim": f, "family": "return"})
    for f in TRIM_FRACTIONS:
        tag = f"f{int(f * 100):02d}"
        SIGNALS.append({"name": f"sharpe{W}_{tag}", "direction": "high", "window": W, "trim": f, "family": "sharpe"})
    SIGNALS.append({"name": f"sortino{W}", "direction": "high", "window": W, "trim": None, "family": "sortino"})
    SIGNALS.append({"name": f"vol{W}", "direction": "low", "window": W, "trim": None, "family": "vol"})
SIGNAL_NAMES = [s["name"] for s in SIGNALS]
SIGNAL_SIGN = {s["name"]: (1.0 if s["direction"] == "high" else -1.0) for s in SIGNALS}
TARGETS = ["fwd60_sharpe", "fwd60_return", "fwd60_vol", "fwd60_log_max_dd", "fwd30_sharpe", "fwd30_return"]
TARGET_SIGN = {"fwd60_sharpe": 1.0, "fwd60_return": 1.0, "fwd60_vol": -1.0, "fwd60_log_max_dd": 1.0, "fwd30_sharpe": 1.0, "fwd30_return": 1.0}
PRIMARY = f"fwd{PRIMARY_H}_sharpe"
#: Raw/trimmed pairs for the MATCHED paired comparison: both Spearmans on the joint finite mask
#: of raw score, trimmed score and primary target (sixth review), so a difference is trimming
#: alone and not a change of eligible vaults.
PAIRS = [(f"{fam}{W}_f00", f"{fam}{W}_f{int(f * 100):02d}") for W in WINDOWS for fam in ("ret", "sharpe") for f in TRIM_FRACTIONS[1:]]
PAIR_INDEX = [(SIGNAL_NAMES.index(a), SIGNAL_NAMES.index(b)) for a, b in PAIRS]
coverage = pd.DataFrame({s: np.isfinite(panel[s]).mean() for s in SIGNAL_NAMES}, index=["finite_share"]).T
display(coverage.round(3).T)


candidate-dates dropped: {'stale_or_too_few_marks': 59998, 'tvl': 25200, 'forward_marks': 858}
panel: 29,912 rows, 397 vaults, 192 decisions 2025-07-01 to 2026-07-18; young (< 360 d from the vault's first mark in the whole archive) share of rows 68.5%; rows from vaults first marked before 2025: 24.3%


,rows,vaults,decisions,events90_median,events180_median,fwd60_events_median,fwd30_events_median,fwd60_finite,fwd30_finite,young_share
regime,,,,,,,,,,
dense Apr 2026 on,12314,341,55,89.0,122.0,60.0,30.0,0.941,0.916,0.647
transition Jan-Mar 2026,8211,248,45,31.0,42.0,60.0,30.0,0.906,0.910,0.654
weekly 2025,9387,185,92,12.0,24.0,9.0,4.0,0.948,0.903,0.761


,ret90_f00,ret90_f10,ret90_f25,sharpe90_f00,sharpe90_f10,sharpe90_f25,sortino90,vol90,ret180_f00,ret180_f10,ret180_f25,sharpe180_f00,sharpe180_f10,sharpe180_f25,sortino180,vol180
finite_share,0.986,0.986,0.986,0.944,0.939,0.938,0.931,0.986,0.693,0.693,0.693,0.675,0.672,0.672,0.67,0.693


## Part 2. One shared bootstrap, every hypothesis

As NB38: per-date signed Spearman, equal-weight mean over dates, one two-way cluster bootstrap
shared by every signal, target and paired difference; simultaneous max-T lower bounds over the
signal family on the primary target and over the paired-comparison family.


In [3]:
def per_date_blocks(frame: pd.DataFrame) -> dict:
    out = {}
    cols = SIGNAL_NAMES + TARGETS
    for date, g in frame.groupby("date"):
        out[pd.Timestamp(date)] = {"vault": g["address"].to_numpy(), "values": g[cols].to_numpy(dtype=float)}
    return out


def _spearman(x: np.ndarray, y: np.ndarray) -> float:
    xr, yr = rankdata(x), rankdata(y)
    if np.ptp(xr) == 0 or np.ptp(yr) == 0:
        return float("nan")
    xc, yc = xr - xr.mean(), yr - yr.mean()
    return float((xc * yc).sum() / math.sqrt((xc ** 2).sum() * (yc ** 2).sum()))


def date_statistics(values: np.ndarray) -> tuple:
    """Per-date signed Spearman for every (signal, target) on each pair's own complete cases, plus
    the MATCHED paired statistics for the primary target: for each (raw, trimmed) pair, both
    Spearmans on the joint finite mask, returned as (raw, trimmed, difference, n)."""
    S, T = len(SIGNAL_NAMES), len(TARGETS)
    out = np.full((S, T), np.nan)
    targets = values[:, S:]
    for i, name in enumerate(SIGNAL_NAMES):
        x = values[:, i]
        for j, target in enumerate(TARGETS):
            y = targets[:, j]
            ok = np.isfinite(x) & np.isfinite(y)
            if ok.sum() < MIN_CANDIDATES:
                continue
            rho = _spearman(x[ok], y[ok])
            if np.isfinite(rho):
                out[i, j] = SIGNAL_SIGN[name] * TARGET_SIGN[target] * rho
    pairs = np.full((len(PAIRS), 4), np.nan)
    jp = TARGETS.index(PRIMARY)
    y = targets[:, jp]
    for k, (ia, ib) in enumerate(PAIR_INDEX):
        xa, xb = values[:, ia], values[:, ib]
        ok = np.isfinite(xa) & np.isfinite(xb) & np.isfinite(y)
        if ok.sum() < MIN_CANDIDATES:
            continue
        ra, rb = _spearman(xa[ok], y[ok]), _spearman(xb[ok], y[ok])
        if np.isfinite(ra) and np.isfinite(rb):
            sign = SIGNAL_SIGN[SIGNAL_NAMES[ia]] * TARGET_SIGN[PRIMARY]
            pairs[k] = (sign * ra, sign * rb, sign * (rb - ra), float(ok.sum()))
    return out, pairs


def mean_over_dates(blocks: dict, dates: list, counts: dict | None = None) -> tuple:
    S, T = len(SIGNAL_NAMES), len(TARGETS)
    total, n = np.zeros((S, T)), np.zeros((S, T))
    ptotal, pn = np.zeros((len(PAIRS), 4)), np.zeros((len(PAIRS), 4))
    for date in dates:
        block = blocks.get(date)
        if block is None:
            continue
        values = block["values"]
        if counts is not None:
            repeats = np.array([counts.get(v, 0) for v in block["vault"]], dtype=int)
            if repeats.sum() < MIN_CANDIDATES:
                continue
            values = np.repeat(values, repeats, axis=0)
        stats, pairs = date_statistics(values)
        finite = np.isfinite(stats)
        total[finite] += stats[finite]
        n[finite] += 1
        pf = np.isfinite(pairs)
        ptotal[pf] += pairs[pf]
        pn[pf] += 1
    with np.errstate(invalid="ignore"):
        return (np.where(n > 0, total / np.maximum(n, 1), np.nan), n,
                np.where(pn > 0, ptotal / np.maximum(pn, 1), np.nan), pn)


def bootstrap(frame: pd.DataFrame, draws: int = DRAWS, seed: int = SEED, verbose: bool = True, block: int = DATE_BLOCK) -> dict:
    """Two-way cluster bootstrap: TILED date blocks and vault clusters, together.

    Per draw the date axis is cut into consecutive tiles of `block` decisions starting at a random
    offset in [0, block) - the first and last tiles are the shorter remainders - and tiles are
    resampled with replacement until the draw holds at least as many decisions as the panel. The
    draw is NOT truncated: the last selected tile is kept whole, so a draw can be slightly longer
    than the panel. Because every decision belongs to exactly one tile and tiles are selected
    with equal probability, every decision has the same expected inclusion count (the equal-weight
    mean over the draw's dates is what the statistic uses). Coverage is measured over the draws
    and asserted below. No tile wraps from the end of the archive to its start.
    """
    blocks = per_date_blocks(frame)
    dates = sorted(blocks)
    vaults = sorted(frame["address"].unique())
    observed, n_dates, pair_observed, pair_n = mean_over_dates(blocks, dates)
    rng = np.random.default_rng(seed)
    n = len(dates)
    block = min(block, n)
    reps = np.full((draws,) + observed.shape, np.nan)
    pair_reps = np.full((draws,) + pair_observed.shape, np.nan)
    inclusion = np.zeros(n)
    draw_lengths = []
    for d in range(draws):
        offset = int(rng.integers(0, block))
        edges = [0] + list(range(offset, n, block)) + [n]
        edges = sorted(set(e for e in edges if 0 <= e <= n))
        tiles = [np.arange(a, b) for a, b in zip(edges[:-1], edges[1:]) if b > a]
        chosen = []
        while sum(len(t) for t in chosen) < n:
            chosen.append(tiles[int(rng.integers(0, len(tiles)))])
        index = np.concatenate(chosen)          # whole tiles only; never truncated
        np.add.at(inclusion, index, 1)
        draw_lengths.append(len(index))
        drawn = rng.choice(len(vaults), size=len(vaults), replace=True)
        counts = {}
        for v in drawn:
            counts[vaults[v]] = counts.get(vaults[v], 0) + 1
        reps[d], _, pair_reps[d], _ = mean_over_dates(blocks, [dates[i] for i in index], counts)
        if verbose and (d + 1) % 100 == 0:
            print(f"  draw {d + 1}/{draws}")
    # Coverage check: with uniform tile selection and no truncation, every date's inclusion count
    # should be equal in expectation. Report the spread across dates relative to the mean and
    # fail if the ends of the archive are systematically under- or over-represented.
    coverage = inclusion / inclusion.mean()
    first, last = float(np.mean(coverage[:block])), float(np.mean(coverage[-block:]))
    ends = float(np.mean(np.concatenate([coverage[:block], coverage[-block:]])))
    middle = float(np.mean(coverage[block:-block])) if n > 2 * block else float("nan")
    print(f"  coverage over {draws} draws: min {coverage.min():.3f}, max {coverage.max():.3f} of mean; first {block} dates {first:.3f}, "
          f"last {block} dates {last:.3f}, middle {middle:.3f}; draw length mean {np.mean(draw_lengths):.1f} of {n} decisions")
    # The first and last blocks are asserted SEPARATELY, so an asymmetric bias cannot cancel in an
    # average; 0.15 of the mean is the tolerance for 500 draws of a random-offset tiling.
    assert abs(first - 1.0) < 0.15 and abs(last - 1.0) < 0.15, f"tiled bootstrap misweights an end: first {first:.3f}, last {last:.3f}"
    return {"observed": observed, "draws": reps, "n_dates": n_dates, "dates": dates, "rows": len(frame),
            "pair_observed": pair_observed, "pair_draws": pair_reps, "pair_n": pair_n,
            "coverage_min": float(coverage.min()), "coverage_max": float(coverage.max()), "coverage_first": first,
            "coverage_last": last, "coverage_ends": ends, "coverage_middle": middle, "draw_length_mean": float(np.mean(draw_lengths))}


def simultaneous_lower(observed: np.ndarray, reps: np.ndarray, level: float = LEVEL) -> dict:
    se = np.nanstd(reps, axis=0, ddof=1)
    member = np.isfinite(observed) & np.isfinite(se) & (se > 0)
    stud = (reps - observed[None, :]) / np.where(se > 0, se, np.nan)[None, :]
    complete = np.isfinite(stud[:, member]).all(axis=1) if member.any() else np.zeros(len(reps), dtype=bool)
    per_draw_max = stud[complete][:, member].max(axis=1) if complete.any() else np.array([])
    critical = float(np.percentile(per_draw_max, level * 100.0)) if len(per_draw_max) >= 100 else float("nan")
    lower = np.where(member, observed - critical * se, np.nan)
    centred = reps - observed[None, :]
    p = (1.0 + (centred >= observed[None, :]).sum(axis=0)) / (len(reps) + 1.0)
    return {"se": se, "critical": critical, "lower": lower, "lower_unadjusted": np.nanpercentile(reps, (1 - level) * 100.0, axis=0),
            "p": p, "family_size": int(member.sum()), "complete_draws": int(len(per_draw_max))}


def screen(frame: pd.DataFrame, label: str, verbose: bool = True, block: int = DATE_BLOCK) -> dict:
    print(f"{label}: {len(frame):,} rows, {frame['date'].nunique()} decisions, {frame['address'].nunique()} vaults, date block {block}")
    boot = bootstrap(frame, verbose=verbose, block=block)
    j = TARGETS.index(PRIMARY)
    fam = simultaneous_lower(boot["observed"][:, j], boot["draws"][:, :, j])
    rows = []
    for i, s in enumerate(SIGNALS):
        row = {"signal": s["name"], "family": s["family"], "window": s["window"], "trim": s["trim"],
               "dates": int(boot["n_dates"][i, j]), "evaluated": bool(boot["n_dates"][i, j] >= MIN_DATES and fam["se"][i] > 0)}
        for t_idx, target in enumerate(TARGETS):
            row[f"rho_{target}"] = boot["observed"][i, t_idx]
        row["se_primary"] = fam["se"][i]
        row["lo_primary_simultaneous"] = fam["lower"][i]
        row["lo_primary_unadjusted"] = fam["lower_unadjusted"][i]
        row["p_primary"] = fam["p"][i]
        rows.append(row)
    table = pd.DataFrame(rows).set_index("signal")
    # MATCHED paired differences: per date, raw and trimmed Spearmans on the joint finite mask of
    # both scores and the primary target, so the difference is trimming alone; resampled on the
    # same draws. The unmatched per-signal correlations in the table above use each score's own
    # complete cases and can differ slightly from the matched values here.
    diffs, d_obs, d_reps = [], [], []
    for k, (raw_name, trim_name) in enumerate(PAIRS):
        fam_name = "ret" if raw_name.startswith("ret") else "sharpe"
        W = int(raw_name.replace(fam_name, "").split("_")[0])
        f = int(trim_name.split("_f")[1]) / 100.0
        obs = boot["pair_observed"][k, 2]
        rep = boot["pair_draws"][:, k, 2]
        d_obs.append(obs); d_reps.append(rep)
        fin = rep[np.isfinite(rep)]; n = len(fin); centred = fin - obs
        p_hi = (1.0 + (centred >= obs).sum()) / (n + 1.0); p_lo = (1.0 + (centred <= obs).sum()) / (n + 1.0)
        diffs.append({"family": fam_name, "window": W, "trim": f,
                      "raw_rho_matched": boot["pair_observed"][k, 0], "trimmed_rho_matched": boot["pair_observed"][k, 1],
                      "raw_rho_unmatched": boot["observed"][SIGNAL_NAMES.index(raw_name), j],
                      "trimmed_rho_unmatched": boot["observed"][SIGNAL_NAMES.index(trim_name), j],
                      "difference": obs, "matched_candidates_mean": boot["pair_observed"][k, 3],
                      "ci_lo": float(np.percentile(fin, 2.5)) if n >= 100 else np.nan,
                      "ci_hi": float(np.percentile(fin, 97.5)) if n >= 100 else np.nan,
                      "p_two_sided_add_one": float(min(1.0, 2 * min(p_hi, p_lo))) if n >= 100 else np.nan, "draws": int(n)})
    paired = pd.DataFrame(diffs)
    pfam = simultaneous_lower(np.array(d_obs), np.column_stack(d_reps))
    paired["lo_simultaneous_family"] = pfam["lower"]
    paired["se"] = pfam["se"]
    return {"label": label, "table": table, "paired": paired, "critical": fam["critical"], "family_size": fam["family_size"],
            "complete_draws": fam["complete_draws"], "boot": boot, "paired_critical": pfam["critical"], "paired_family_size": pfam["family_size"],
            "block": block}


TABLE_COLS = ["family", "window", "trim", "dates", "evaluated", f"rho_{PRIMARY}", "se_primary", "lo_primary_simultaneous",
              "lo_primary_unadjusted", "p_primary", "rho_fwd60_return", "rho_fwd60_vol", "rho_fwd60_log_max_dd", "rho_fwd30_sharpe", "rho_fwd30_return"]
manifest_38 = json.loads(Path("_build/manifest_38.json").read_text())
nb38_ref = {"nb38_decisions": manifest_38["panel"]["decisions"], "nb38_rows": manifest_38["panel"]["rows"],
            "nb38_sharpe180_rho_fwd30_sharpe": manifest_38["screens"]["all"]["table"]["sharpe180_k0"]["rho_fwd_sharpe"],
            "nb38_sharpe180_lo_simultaneous": manifest_38["screens"]["all"]["table"]["sharpe180_k0"]["lo_sharpe_simultaneous"]}
print("NB38 comparison values (from _build/manifest_38.json):", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in nb38_ref.items()})
full = screen(panel, "all regimes")
print(f"\nprimary family: {full['family_size']} signals, critical {full['critical']:.4f} on {full['complete_draws']} complete draws")
display(full["table"][TABLE_COLS].round(4))
# Sensitivity to the block length: 45 decisions (90 days) as an intermediate, 90 decisions (180 days) as the
# longest trailing-score window.
full45 = screen(panel, "all regimes, block 45", verbose=False, block=DATE_BLOCK_SENSITIVITY)
full90 = screen(panel, "all regimes, block 90", verbose=False, block=DATE_BLOCK_LONG)
sens = pd.DataFrame({"rho": full["table"][f"rho_{PRIMARY}"], "lo_block30": full["table"]["lo_primary_simultaneous"],
                     "lo_block45": full45["table"]["lo_primary_simultaneous"], "lo_block90": full90["table"]["lo_primary_simultaneous"],
                     "se_block30": full["table"]["se_primary"], "se_block45": full45["table"]["se_primary"], "se_block90": full90["table"]["se_primary"]})
for b in (30, 45, 90):
    sens[f"clears_block{b}"] = sens[f"lo_block{b}"] > 0
print(f"\nblock-length sensitivity (critical {full['critical']:.3f} at 30, {full45['critical']:.3f} at 45, {full90['critical']:.3f} at 90 decisions): "
      f"{int(sens['clears_block30'].sum())} signals clear at block 30, {int(sens['clears_block45'].sum())} at 45, {int(sens['clears_block90'].sum())} at 90 "
      f"(90 decisions = 180 days = the longest trailing window; the {len(full['boot']['dates'])} decisions are "
      f"{len(full['boot']['dates']) / DATE_BLOCK_LONG:.2f} block-equivalents; the source tiling is one or two full tiles plus "
      f"remainders depending on the offset, and a draw repeats tiles)")
display(sens.round(4))
print(f"\nMATCHED PAIRED trimmed - raw on {PRIMARY}, both Spearmans on the joint finite mask per date (per-comparison 95% "
      f"intervals; simultaneous lower bound over the {full['paired_family_size']} paired comparisons, critical {full['paired_critical']:.4f}):")
display(full["paired"].round(4))


NB38 comparison values (from _build/manifest_38.json): {'nb38_decisions': 70, 'nb38_rows': 17197, 'nb38_sharpe180_rho_fwd30_sharpe': 0.1356, 'nb38_sharpe180_lo_simultaneous': -0.0132}
all regimes: 29,912 rows, 192 decisions, 397 vaults, date block 30


  draw 100/500


  draw 200/500


  draw 300/500


  draw 400/500


...

  coverage over 500 draws: min 0.891, max 1.090 of mean; first 45 dates 1.026, last 45 dates 1.047, middle 0.968; draw length mean 208.6 of 192 decisions
all regimes, block 90: 29,912 rows, 192 decisions, 397 vaults, date block 90


  coverage over 500 draws: min 0.936, max 1.092 of mean; first 90 dates 1.001, last 90 dates 1.002, middle 0.980; draw length mean 221.5 of 192 decisions

block-length sensitivity (critical 2.281 at 30, 2.387 at 45, 2.465 at 90 decisions): 15 signals clear at block 30, 15 at 45, 15 at 90 (90 decisions = 180 days = the longest trailing window; the 192 decisions are 2.13 block-equivalents; the source tiling is one or two full tiles plus remainders depending on the offset, and a draw repeats tiles)


,rho,lo_block30,lo_block45,lo_block90,se_block30,se_block45,se_block90,clears_block30,clears_block45,clears_block90
signal,,,,,,,,,,
ret90_f00,0.1513,0.0248,0.0233,0.0274,0.0555,0.0536,0.0503,True,True,True
ret90_f10,0.2110,0.0948,0.0954,0.0913,0.0509,0.0484,0.0485,True,True,True
ret90_f25,0.2092,0.0902,0.0907,0.0871,0.0522,0.0497,0.0495,True,True,True
sharpe90_f00,0.2123,0.0895,0.0882,0.0836,0.0538,0.0520,0.0522,True,True,True
sharpe90_f10,0.1994,0.0785,0.0694,0.0619,0.0530,0.0545,0.0558,True,True,True
sharpe90_f25,0.1757,0.0430,0.0280,0.0176,0.0582,0.0619,0.0642,True,True,True
sortino90,0.2006,0.0820,0.0812,0.0789,0.0520,0.0500,0.0494,True,True,True
vol90,0.1600,0.0503,0.0534,0.0487,0.0481,0.0447,0.0451,True,True,True
ret180_f00,0.2033,0.0774,0.0697,0.0672,0.0552,0.0560,0.0552,True,True,True



MATCHED PAIRED trimmed - raw on fwd60_sharpe, both Spearmans on the joint finite mask per date (per-comparison 95% intervals; simultaneous lower bound over the 8 paired comparisons, critical 2.3312):


,family,window,trim,raw_rho_matched,trimmed_rho_matched,raw_rho_unmatched,trimmed_rho_unmatched,difference,matched_candidates_mean,ci_lo,ci_hi,p_two_sided_add_one,draws,lo_simultaneous_family,se
0,ret,90,0.10,0.1513,0.2110,0.1513,0.2110,0.0596,143.3073,-0.0106,0.1355,0.1158,500,-0.0246,0.0361
1,ret,90,0.25,0.1513,0.2092,0.1513,0.2092,0.0579,143.3073,-0.0254,0.1490,0.1677,500,-0.0431,0.0433
2,sharpe,90,0.10,0.2113,0.1994,0.2123,0.1994,-0.0119,141.1146,-0.0622,0.0354,0.5469,500,-0.0705,0.0251
3,sharpe,90,0.25,0.2117,0.1757,0.2123,0.1757,-0.0359,141.0208,-0.1161,0.0392,0.3273,500,-0.1264,0.0388
4,ret,180,0.10,0.2033,0.2132,0.2033,0.2132,0.0099,99.9427,-0.0497,0.0767,0.7784,500,-0.0631,0.0314
5,ret,180,0.25,0.2033,0.1962,0.2033,0.1962,-0.0071,99.9427,-0.0881,0.0761,0.8303,500,-0.1047,0.0418
6,sharpe,180,0.10,0.2506,0.2567,0.2505,0.2567,0.0061,99.2083,-0.0484,0.0631,0.8343,500,-0.0589,0.0279
7,sharpe,180,0.25,0.2506,0.2191,0.2505,0.2191,-0.0314,99.2083,-0.1267,0.0565,0.4391,500,-0.1383,0.0459


### Reachability

A noisy foresight oracle (the primary target plus 5% noise) through the identical machinery must
clear the family-wise lower bound; otherwise an all-fail result says nothing about the signals.


In [4]:
rng = np.random.default_rng(SEED + 1)
oracle_panel = panel.copy()
oracle_panel["oracle"] = oracle_panel[PRIMARY] + rng.normal(0.0, 0.05 * float(np.nanstd(oracle_panel[PRIMARY])), len(oracle_panel))
_saved = (SIGNALS, SIGNAL_NAMES, SIGNAL_SIGN)
SIGNALS = list(SIGNALS) + [{"name": "oracle", "direction": "high", "window": None, "trim": None, "family": "oracle"}]
SIGNAL_NAMES = [s["name"] for s in SIGNALS]
SIGNAL_SIGN = {s["name"]: (1.0 if s["direction"] == "high" else -1.0) for s in SIGNALS}
try:
    oracle_res = screen(oracle_panel, "oracle reachability", verbose=False)
finally:
    SIGNALS, SIGNAL_NAMES, SIGNAL_SIGN = _saved
orow = oracle_res["table"].loc["oracle"]
print(f"oracle: rho {orow[f'rho_{PRIMARY}']:.4f}, simultaneous lower bound {orow['lo_primary_simultaneous']:.4f} over "
      f"{oracle_res['family_size']} signals (critical {oracle_res['critical']:.4f}); finite primary target rows "
      f"{int(np.isfinite(panel[PRIMARY]).sum())} of {len(panel)}")
assert orow["lo_primary_simultaneous"] > 0, "the screen cannot produce a positive simultaneous bound even for a foresight oracle"
print("reachable")


oracle reachability: 29,912 rows, 192 decisions, 397 vaults, date block 30


  coverage over 500 draws: min 0.923, max 1.107 of mean; first 30 dates 1.015, last 30 dates 1.000, middle 0.997; draw length mean 203.3 of 192 decisions
oracle: rho 0.9975, simultaneous lower bound 0.9965 over 17 signals (critical 2.2812); finite primary target rows 27932 of 29912
reachable


## Part 3. Per regime, and young against old

A regime cohort is the set of decisions whose decision date AND whole 60-day forward horizon lie
inside the regime, so a weekly-regime outcome is measured on weekly marks. Each cohort is a
separate sample with a separate bootstrap. Young (< 360 days) and old are split on the whole
panel; they have different date coverage and are estimates on different samples, not a test of
a difference.


In [5]:
SHORT_COLS = ["window", "trim", "dates", "evaluated", f"rho_{PRIMARY}", "lo_primary_simultaneous", "p_primary",
              "rho_fwd60_return", "rho_fwd60_vol", "rho_fwd30_sharpe"]
by_regime_screens = {}
print("decisions whose whole 60-day horizon lies inside one regime:", panel.groupby("regime_contained")["date"].nunique().to_dict())
for name, a, b in REGIMES:
    sub = panel[panel["regime_contained"] == name]
    if sub["date"].nunique() < MIN_DATES:
        print(f"{name}: only {sub['date'].nunique()} decisions with the whole horizon inside the regime - not screened")
        continue
    res = screen(sub, name, verbose=False)
    by_regime_screens[name] = res
    print(f"  family {res['family_size']}, critical {res['critical']:.4f}, complete draws {res['complete_draws']}")
    display(res["table"][SHORT_COLS].round(4))
    print(f"  paired trimmed - raw on {PRIMARY} (family critical {res['paired_critical']:.4f}):")
    display(res["paired"].round(4))
young = screen(panel[panel["young"]], "young (< 360 days)", verbose=False)
old = screen(panel[~panel["young"]], "old (>= 360 days)", verbose=False)
for res in (young, old):
    print(f"\n{res['label']}: family {res['family_size']}, critical {res['critical']:.4f}")
    display(res["table"][SHORT_COLS].round(4))
    display(res["paired"].round(4))


decisions whose whole 60-day horizon lies inside one regime: {'crosses': 60, 'dense Apr 2026 on': 55, 'transition Jan-Mar 2026': 15, 'weekly 2025': 62}
weekly 2025: 5,237 rows, 62 decisions, 138 vaults, date block 30


  coverage over 500 draws: min 0.942, max 1.035 of mean; first 30 dates 1.015, last 30 dates 0.986, middle 0.993; draw length mean 72.1 of 62 decisions
  family 16, critical 2.2728, complete draws 500


,window,trim,dates,evaluated,rho_fwd60_sharpe,lo_primary_simultaneous,p_primary,rho_fwd60_return,rho_fwd60_vol,rho_fwd30_sharpe
signal,,,,,,,,,,
ret90_f00,90,0.00,62,True,0.1485,-0.0382,0.0419,0.1423,0.0004,0.1202
ret90_f10,90,0.10,62,True,0.2402,0.0748,0.0020,0.2164,0.3560,0.2031
ret90_f25,90,0.25,62,True,0.2435,0.0771,0.0020,0.2209,0.4505,0.2121
sharpe90_f00,90,0.00,62,True,0.2502,0.0752,0.0020,0.2034,0.1544,0.2159
sharpe90_f10,90,0.10,62,True,0.2730,0.1034,0.0020,0.2086,0.1219,0.2390
sharpe90_f25,90,0.25,62,True,0.2798,0.1132,0.0020,0.2179,0.1433,0.2436
sortino90,90,NaN,62,True,0.2336,0.0581,0.0020,0.1963,0.1667,0.2010
vol90,90,NaN,62,True,0.1632,-0.0223,0.0180,0.1477,0.6850,0.1551
ret180_f00,180,0.00,62,True,0.2496,0.0452,0.0020,0.2405,0.2460,0.1879


  paired trimmed - raw on fwd60_sharpe (family critical 2.3602):


,family,window,trim,raw_rho_matched,trimmed_rho_matched,raw_rho_unmatched,trimmed_rho_unmatched,difference,matched_candidates_mean,ci_lo,ci_hi,p_two_sided_add_one,draws,lo_simultaneous_family,se
0,ret,90,0.10,0.1485,0.2402,0.1485,0.2402,0.0917,79.3710,-0.0133,0.1915,0.0679,500,-0.0333,0.0530
1,ret,90,0.25,0.1485,0.2435,0.1485,0.2435,0.0951,79.3710,-0.0237,0.2116,0.1158,500,-0.0542,0.0632
2,sharpe,90,0.10,0.2489,0.2730,0.2502,0.2730,0.0241,78.1452,-0.0297,0.0848,0.3393,500,-0.0439,0.0288
3,sharpe,90,0.25,0.2510,0.2798,0.2502,0.2798,0.0288,77.9839,-0.0309,0.1034,0.3353,500,-0.0502,0.0335
4,ret,180,0.10,0.2496,0.2667,0.2496,0.2667,0.0171,52.7903,-0.0612,0.1041,0.6627,500,-0.0842,0.0429
5,ret,180,0.25,0.2496,0.2425,0.2496,0.2425,-0.0071,52.7903,-0.1323,0.1368,0.9421,500,-0.1697,0.0689
6,sharpe,180,0.10,0.2808,0.3054,0.2808,0.3054,0.0247,52.3871,-0.0491,0.0949,0.4631,500,-0.0630,0.0371
7,sharpe,180,0.25,0.2808,0.2858,0.2808,0.2858,0.0051,52.3871,-0.1288,0.1194,0.9222,500,-0.1412,0.0620


...

,window,trim,dates,evaluated,rho_fwd60_sharpe,lo_primary_simultaneous,p_primary,rho_fwd60_return,rho_fwd60_vol,rho_fwd30_sharpe
signal,,,,,,,,,,
ret90_f00,90,0.00,192,True,0.1477,0.0106,0.0060,0.1261,0.1048,0.1040
ret90_f10,90,0.10,192,True,0.2319,0.1116,0.0020,0.2049,0.4656,0.1954
ret90_f25,90,0.25,192,True,0.2326,0.1111,0.0020,0.2110,0.5357,0.1971
sharpe90_f00,90,0.00,192,True,0.2083,0.0735,0.0020,0.1453,0.1322,0.1617
sharpe90_f10,90,0.10,192,True,0.2094,0.0818,0.0020,0.1505,0.1647,0.1715
sharpe90_f25,90,0.25,192,True,0.1974,0.0652,0.0020,0.1500,0.2396,0.1615
sortino90,90,NaN,192,True,0.1959,0.0638,0.0020,0.1387,0.1364,0.1472
vol90,90,NaN,192,True,0.1816,0.0586,0.0020,0.1726,0.6355,0.1636
ret180_f00,180,0.00,192,True,0.2205,0.0657,0.0020,0.1854,0.2073,0.1683


,family,window,trim,raw_rho_matched,trimmed_rho_matched,raw_rho_unmatched,trimmed_rho_unmatched,difference,matched_candidates_mean,ci_lo,ci_hi,p_two_sided_add_one,draws,lo_simultaneous_family,se
0,ret,90,0.10,0.1477,0.2319,0.1477,0.2319,0.0843,100.1094,0.0122,0.1645,0.0359,500,-0.0010,0.0384
1,ret,90,0.25,0.1477,0.2326,0.1477,0.2326,0.0849,100.1094,0.0010,0.1766,0.0719,500,-0.0152,0.0451
2,sharpe,90,0.10,0.2071,0.2094,0.2083,0.2094,0.0022,98.6719,-0.0579,0.0535,0.9701,500,-0.0599,0.0280
3,sharpe,90,0.25,0.2075,0.1974,0.2083,0.1974,-0.0101,98.5781,-0.0983,0.0649,0.7864,500,-0.1013,0.0411
4,ret,180,0.10,0.2205,0.2357,0.2205,0.2357,0.0152,54.5781,-0.0608,0.0800,0.6347,500,-0.0671,0.0371
5,ret,180,0.25,0.2205,0.2191,0.2205,0.2191,-0.0014,54.5781,-0.0955,0.0888,0.9741,500,-0.1149,0.0512
6,sharpe,180,0.10,0.2412,0.2571,0.2410,0.2571,0.0158,54.2969,-0.0656,0.0820,0.6547,500,-0.0642,0.0361
7,sharpe,180,0.25,0.2412,0.2360,0.2410,0.2360,-0.0052,54.2969,-0.1050,0.0911,0.8782,500,-0.1202,0.0519



old (>= 360 days): family 16, critical 2.2700


,window,trim,dates,evaluated,rho_fwd60_sharpe,lo_primary_simultaneous,p_primary,rho_fwd60_return,rho_fwd60_vol,rho_fwd30_sharpe
signal,,,,,,,,,,
ret90_f00,90,0.00,191,True,0.1216,-0.0693,0.0459,0.1593,0.1784,0.1071
ret90_f10,90,0.10,191,True,0.1223,-0.0931,0.0798,0.1858,0.5890,0.1228
ret90_f25,90,0.25,191,True,0.1064,-0.1200,0.1357,0.1793,0.6766,0.1050
sharpe90_f00,90,0.00,191,True,0.2053,-0.0040,0.0100,0.1619,0.1555,0.1826
sharpe90_f10,90,0.10,191,True,0.1664,-0.0419,0.0220,0.1240,0.1854,0.1569
sharpe90_f25,90,0.25,191,True,0.1135,-0.1153,0.1118,0.0922,0.2844,0.1262
sortino90,90,NaN,191,True,0.1910,-0.0156,0.0140,0.1580,0.1716,0.1671
vol90,90,NaN,191,True,0.0676,-0.1855,0.2575,0.1507,0.7489,0.0727
ret180_f00,180,0.00,192,True,0.1203,-0.0632,0.0599,0.1516,0.3438,0.1194


,family,window,trim,raw_rho_matched,trimmed_rho_matched,raw_rho_unmatched,trimmed_rho_unmatched,difference,matched_candidates_mean,ci_lo,ci_hi,p_two_sided_add_one,draws,lo_simultaneous_family,se
0,ret,90,0.10,0.1216,0.1223,0.1216,0.1223,0.0007,43.3874,-0.1292,0.1368,0.9900,500,-0.1579,0.0700
1,ret,90,0.25,0.1216,0.1064,0.1216,0.1064,-0.0152,43.3874,-0.1725,0.1550,0.8184,500,-0.2146,0.0879
2,sharpe,90,0.10,0.2052,0.1664,0.2053,0.1664,-0.0388,42.6283,-0.1201,0.0308,0.2675,500,-0.1259,0.0384
3,sharpe,90,0.25,0.2052,0.1135,0.2053,0.1135,-0.0917,42.6283,-0.2419,0.0247,0.1397,500,-0.2377,0.0644
4,ret,180,0.10,0.1203,0.1227,0.1203,0.1227,0.0024,45.3646,-0.0978,0.1064,0.9860,500,-0.1193,0.0537
5,ret,180,0.25,0.1203,0.1172,0.1203,0.1172,-0.0030,45.3646,-0.1402,0.1566,0.9381,500,-0.1744,0.0756
6,sharpe,180,0.10,0.2378,0.2229,0.2381,0.2229,-0.0149,44.9115,-0.0833,0.0536,0.5709,500,-0.1001,0.0376
7,sharpe,180,0.25,0.2378,0.1725,0.2381,0.1725,-0.0653,44.9115,-0.1939,0.0666,0.2555,500,-0.2187,0.0677


## Part 4. Manifest


In [6]:
def table_records(res):
    return {"table": res["table"].round(6).to_dict(orient="index"), "paired": res["paired"].round(6).to_dict(orient="records"),
            "critical": res["critical"], "family_size": res["family_size"], "complete_draws": res["complete_draws"],
            "paired_critical": res["paired_critical"], "paired_family_size": res["paired_family_size"],
            "rows": int(res["boot"]["rows"]), "decisions": int(len(res["boot"]["dates"])), "block": res["block"],
            "coverage": {k: res["boot"][k] for k in ("coverage_min", "coverage_max", "coverage_first", "coverage_last", "coverage_ends", "coverage_middle", "draw_length_mean")}}

manifest = {
    "verdict": "DIAGNOSTIC - a vault-level screen on the full archive, not a result",
    "provenance": {**PROVENANCE, "last_mark": str(LAST_MARK)},
    "constants": {"windows": list(WINDOWS), "trim_fractions": list(TRIM_FRACTIONS), "horizons": {str(k): v for k, v in HORIZONS.items()},
                  "primary": PRIMARY, "panel_start": str(PANEL_START.date()), "min_tvl_usd": MIN_TVL_USD, "min_events": MIN_EVENTS,
                  "stale_days": STALE_DAYS, "min_candidates": MIN_CANDIDATES, "min_dates": MIN_DATES, "young_days": YOUNG_DAYS,
                  "draws": DRAWS, "date_block": DATE_BLOCK, "date_block_sensitivity": DATE_BLOCK_SENSITIVITY, "date_block_long": DATE_BLOCK_LONG, "seed": SEED},
    "regimes": [(n, str(a.date()), str(b.date())) for n, a, b in REGIMES],
    "density": dens.round(6).reset_index().astype({"month": str}).to_dict(orient="records"),
    "panel": {"rows": int(len(panel)), "vaults": int(panel["address"].nunique()), "decisions": int(panel["date"].nunique()),
              "first": str(panel["date"].min().date()), "last": str(panel["date"].max().date()), "young_share": float(panel["young"].mean())},
    "dropped": dropped,
    "by_regime": by_regime.round(6).to_dict(orient="index"),
    "coverage": coverage["finite_share"].round(6).to_dict(),
    "screens": {"all": table_records(full), "all_block45": table_records(full45), "all_block90": table_records(full90),
                **{n: table_records(r) for n, r in by_regime_screens.items()},
                "young": table_records(young), "old": table_records(old)},
    "block_sensitivity": sens.round(6).to_dict(orient="index"),
    "regime_contained_decisions": {k: int(v) for k, v in panel.groupby("regime_contained")["date"].nunique().to_dict().items()},
    "nb38_reference": nb38_ref,
    "oracle": {"rho": float(orow[f"rho_{PRIMARY}"]), "lo_simultaneous": float(orow["lo_primary_simultaneous"]),
               "family_size": oracle_res["family_size"], "critical": oracle_res["critical"]},
}
Path("_build/manifest_39.json").write_text(json.dumps(manifest, indent=1, default=str))
print("wrote _build/manifest_39.json")


wrote _build/manifest_39.json
